# Regex vs LLM Sonuç Çıkarımı Karşılaştırması
Bu notebook, veri setinden regex ile başarıyla çıkarılmış 100 kararı alır ve lokal bir LLM kullanarak da aynı işlemi yapıp kıyaslamanızı sağlar.

In [ ]:
import re
import pandas as pd
from datasets import load_from_disk
import random
import time
import warnings
warnings.filterwarnings('ignore')

## 1. Veri Setini Yükleme ve Regex Uygulama
Burada regex mantığını tanımlıyor ve veri setinden 100 rastgele örnek seçiyoruz.

In [ ]:
# Regex Kalıpları
STRICT_PATTERNS = [
    re.compile(r'\b(?:H\s*Ü\s*K\s*Ü\s*M|S\s*O\s*N\s*U\s*Ç)\s*(?:[:：]|[\n\r]+)', re.IGNORECASE),
    re.compile(r'\bG\s*E\s*R\s*E\s*Ğ\s*İ\s+D\s*Ü\s*Ş\s*Ü\s*N\s*Ü\s*L\s*D\s*Ü\s*(?:[:：]|[\n\r]+)', re.IGNORECASE),
]

def extract_verdict_pure(text):
    normalized = text.replace('\xa0', ' ').replace('**', '')
    normalized = normalized.replace('\r\n', '\n').replace('\r', '\n')
    normalized = re.sub(r'[ \t]+', ' ', normalized)
    text_length = len(normalized)
    
    for pattern in STRICT_PATTERNS:
        matches = list(pattern.finditer(normalized))
        valid_matches = [m for m in matches if m.start() > (text_length * 0.5)]
        if valid_matches:
            last_match = valid_matches[-1]
            verdict_text = normalized[last_match.end():].strip()
            if len(verdict_text) > 25: 
                return verdict_text
    return None

# Veri setini yükle
print("Veri seti yükleniyor...")
ds = load_from_disk(r'C:\work environment\Python\nlp_exercises\saved_dataset')
df = ds['train'].to_pandas()

# Regex işlemini uygula
print("Regex işlemi uygulanıyor...")
df['hukum_text_regex'] = df['text'].apply(extract_verdict_pure)

# Sadece başarılı çıkarımları filtrele
successful_extractions = df[df['hukum_text_regex'].notna()]

# Rastgele 100 örnek seç
sample_size = min(100, len(successful_extractions))
df_sample = successful_extractions.sample(n=sample_size, random_state=42).copy()
print(f"Rastgele {sample_size} örnek seçildi.")

## 2. Lokal LLM Kurulumu (Ollama - Qwen2.5:7b)
Bilgisayarınızda Ollama kurulu olduğu için doğrudan `requests` kütüphanesi ile Ollama'nın yerel API'sine bağlanacağız.

In [ ]:
import requests

def extract_verdict_with_llm(text):
    """
    Ollama üzerinden lokal Qwen2.5:7b modelini çağırır.
    """
    prompt = f"Aşağıdaki hukuki metnin sadece hüküm (sonuç/karar) kısmını aynen yaz. Ekstra hiçbir yorum veya açıklama ekleme:\n\n{text[-2000:]}"
    try:
        response = requests.post("http://localhost:11434/api/generate", json={
            "model": "qwen2.5:7b",
            "prompt": prompt,
            "stream": False
        })
        if response.status_code == 200:
            return response.json().get('response', '').strip()
        else:
            return f"[HATA: {response.status_code}]"
    except Exception as e:
        return f"[Ollama Bağlantı Hatası: Lütfen Ollama'nın arka planda çalıştığından emin olun. Hata: {str(e)}]"

## 3. LLM'i Çalıştırma ve Karşılaştırma
Tüm örnekler için LLM fonksiyonunu çalıştırıp, sonuçları bir tablo halinde görüyoruz.

In [ ]:
print("LLM çıkarma işlemi başlatılıyor...")
llm_results = []
for idx, row in df_sample.iterrows():
    # LLM çok uzun metinlerde bağlamı kaybetmesin veya token limitini aşmasın diye
    # sadece karar kısmını (metnin son 1500-2000 karakteri) göndermek daha mantıklıdır.

    llm_output = extract_verdict_with_llm(row['text'])
    llm_results.append(llm_output)
    
df_sample['hukum_text_llm'] = llm_results
print("LLM işlemi tamamlandı!")

In [ ]:
# Orijinal metnin son 500 karakterini önizleme amaçlı kesiyoruz
df_sample['text_tail_preview'] = df_sample['text'].apply(lambda x: "..." + x[-500:].replace('\n', ' '))

# Karşılaştırma Dataframe'ini oluşturuyoruz
df_comparison = df_sample[['id', 'text_tail_preview', 'hukum_text_regex', 'hukum_text_llm']].copy()

# Tabloyu Notebook'ta tam görebilmek için pandas görünüm ayarları
pd.set_option('display.max_colwidth', None)

# İlk 5 örneği ekrana yazdır
df_comparison.head()

In [ ]:
# Tüm 100 kıyaslamayı Excel dosyası olarak kaydet (İncelemek için en iyi yöntem)
output_file = "regex_vs_llm_karsilastirma.xlsx"
df_comparison.to_excel(output_file, index=False)
print(f"Karşılaştırma sonuçları başarıyla '{output_file}' dosyasına kaydedildi!")